In [107]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd

# Import du processeur de production spécialisé
from tools.OI_class_OP import OI_ProductionProcessor
from tools.OI_Dashboard import ProductionDashboard
from tools.OI_Dashboard_v2 import AjouterVisualisationsAvancees


# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [108]:
# Cellule 2 : Définition des métadonnées de tags API
tags = [
    {'tag':'WQ33222VA', 'nom':'ester_cons','info':'Totalisation du Peson Acetate/Propionate', 'agg': 'FIRST' },
    {'tag':'NOP_ESTERS', 'nom':'ester_nop','info':"Nombres des opérations d'esters",'agg':'FIRST' },
    {'tag':'3340_type', 'nom':'A/P','info':'Acetate ou Propionate','agg':'FIRST' },
    {'tag':'CTY_ACV43A_Teneur Vit. A (UV)', 'nom':'Acetate_UV','info':'ACV43A teneur en acetate', 'agg': 'MEAN'},
    {'tag':'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom':'Propionate_UV', 'info':'A3340FGB teneur en propionate', 'agg': 'MEAN'},
    {'tag':'PU3310VA_Sign','nom':'PU3310','info':'signature du PU3310','agg':'FIRST'},
    {'tag':'PU3320VA_Sign','nom':'PU3320','info':'signature du PU3320','agg':'FIRST'},
    {'tag':'PU3340VA_Sign','nom':'PU3340','info':'signature du PU3340','agg':'FIRST'},
    {'tag':'FQ32202VA_UV','nom':'Hexane','info':'VA diluée dans de l\'hexane', 'agg': 'FIRST'},
    {'tag':'CTY_A3230A_Teneur en rétinol', 'nom':'Retinol_UV','info':'A3230A teneur en rétinol lavé', 'agg': 'MEAN'},
    {'tag':'LI33203VA','nom':'R33020','info':'niveau du R33020', 'agg': 'FIRST'},
    {'tag':'LI33218VA','nom':'R33022','info':'niveau du R33022', 'agg': 'FIRST'},
    {'tag':'LI33225VA','nom':'R33061','info':'niveau du R33061', 'agg': 'FIRST'},
    {'tag':'WI33222VA','nom':'R33060','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'FQ32202VA','nom':'retinol_cons','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'LI32209VA','nom':'R32031','info':'niveau du R32031', 'agg': 'FIRST'},
]

In [109]:
# Cellule 3 : Variable Produits unifiée (Acetate & Propionate)
# Plus aucune distinction batch / continu pour le calcul global du stock d'un produit.
produits = [
    {
        'nom': 'Ester',
        'conso': {
            'value': 'ester_cons',
            'scale': 1e-9,
            'type':'A/P',
            'uv': ['Acetate_UV', 'Propionate_UV'],
        },
        'CMJ': 9.5,
        'NOP': {
            'value':'NOP_ESTERS',
            'scale': 1,
            'type': None,
        },
        'stock': [
            # Batchs
            {'pu': 'PU3310', 'in': 540, 'out': 710, 'value': 'Hexane', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
            {'pu': 'PU3310', 'in': 710, 'out': 2320, 'value': None, 'uv': None, 'scale': 2.71},
            {'pu': 'PU3320', 'in': 430, 'out': 2020, 'value': None, 'uv': None, 'scale': 2.71},
            # Continus
            {'pu': None, 'value': 'R33020', 'min': 14, 'epalage': [[28.62, 21.22, 3.866, -0.0983], [-228.84, 74.557]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33022', 'min': 14, 'epalage': [[6.25, 5.4525, 0.898, -0.0256], [-35, 97, 16.039]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33061', 'min': 18, 'epalage': [[0.69, 0.72, 0.296, -0.0059], [-30.03, 5.836]], 'uv': None, 'scale': 0.95 , 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': 'PU3340', 'in': 0, 'out': 710, 'value': 'R33060', 'uv': None, 'scale': 0.95, 'cond': 'A/P', 'val_cond': [1/344, 1/359]}
        ]
    },
    {
        'nom': 'Retinol',
        'conso': {
            'value':'retinol_cons',
            'scale': 1e-5, # Ajustement de l'échelle pour le retinol g et analyse en %
            'uv': ['Retinol_UV','Retinol_UV'],
        },
        'CMJ': 1,
        'stock': [
            # Batchs
            {'pu': None, 'value': 'R32031', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
        ]
    }
]

In [110]:
# Cellule 4 : Initialisation du processeur de production spécialisé
processor = OI_ProductionProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start = '2026-01-01',
    end = '2026-12-31',
    tags_metadata = tags,
    produits = produits,
    interval = 'PT10M',
    verbose = False
)

In [111]:
(processor.agg_mapping)

{'WQ33222VA': 'FIRST',
 'NOP_ESTERS': 'FIRST',
 '3340_type': 'FIRST',
 'CTY_ACV43A_Teneur Vit. A (UV)': 'MEAN',
 'CTY_A3340FGB_Teneur arr. Vit. A (UV)': 'MEAN',
 'PU3310VA_Sign': 'FIRST',
 'PU3320VA_Sign': 'FIRST',
 'PU3340VA_Sign': 'FIRST',
 'FQ32202VA_UV': 'FIRST',
 'CTY_A3230A_Teneur en rétinol': 'MEAN',
 'LI33203VA': 'FIRST',
 'LI33218VA': 'FIRST',
 'LI33225VA': 'FIRST',
 'WI33222VA': 'FIRST',
 'FQ32202VA': 'FIRST',
 'LI32209VA': 'FIRST'}

In [112]:
# Cellule 5 : Téléchargement et calcul automatique des bilans par produit
processor.merge()
processor.compute_production_balance()

# Visualisation des premières lignes calculées
processor.data.describe()

,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester,consommation_Retinol,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
count,2.315400e+04,23154.000000,23154.000000,2.315400e+04,2.315400e+04,23142.000000,23145.000000,23141.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000,23154.000000
mean,2.466678e+06,1974.850096,0.071753,2.226448e+06,2.275779e+06,1030.416559,963.641305,460.892788,2609.415847,20.778479,46.255533,4.959362,7.252708,689.257470,482116.448468,47.803280,547.036757,6.503251,428766.540603,0.012038,547.036757,0.916964,547.953721,428766.540603,0.000483,428766.541086
std,1.539694e+05,126.280027,0.258064,3.753304e+04,5.434295e+04,622.528435,553.769208,119.135273,1880.830152,0.361492,20.405451,2.471297,11.515101,353.313932,256199.354484,21.042983,344.702562,2.155310,494225.181326,0.005301,344.702562,2.155310,345.318830,494225.181326,0.005301,494225.181619
min,2.221310e+06,1774.000000,0.000000,2.078915e+06,2.188000e+06,100.000000,100.000000,100.000000,0.000000,20.000000,-8.417280,-0.695313,-0.536877,4.790160,480.250000,-1.354620,0.000000,0.702392,0.000000,-0.000342,0.000000,-4.883895,-1.990054,0.000000,-0.011897,0.000000
25%,2.319790e+06,1855.000000,0.000000,2.221983e+06,2.242000e+06,400.000000,400.000000,410.000000,0.000000,20.600000,24.470825,3.073342,2.243982,424.775250,263744.000000,39.882975,217.947562,5.397375,69.542879,0.010008,217.947562,-0.188912,218.412744,69.542879,-0.001548,69.545704
50%,2.455020e+06,1964.000000,0.000000,2.231260e+06,2.300000e+06,1310.000000,1110.000000,410.000000,3961.610000,20.800000,46.081400,5.992190,3.025505,684.791000,462931.000000,45.100200,519.124082,6.789394,161.855269,0.011512,519.124082,1.203107,516.909322,161.855269,-0.000044,161.858593
75%,2.597620e+06,2082.000000,0.000000,2.244406e+06,2.314000e+06,1310.000000,1555.000000,410.000000,4043.440000,21.200000,65.037825,6.566410,4.035160,949.361500,707159.000000,67.183600,839.640009,7.985343,998522.530859,0.016849,839.640009,2.399055,841.905515,998522.530859,0.005294,998522.534772
max,2.753260e+06,2210.000000,1.000000,2.325113e+06,2.626000e+06,2340.000000,2290.000000,810.000000,4836.070000,21.200000,91.609100,12.334800,74.049600,1322.300000,998943.000000,79.734400,1189.939981,12.323393,998627.995895,0.019909,1189.939981,6.737105,1192.504342,998627.995895,0.008354,998628.002986


In [113]:
# ✨ INITIALISER LE DASHBOARD ✨
dashboard = ProductionDashboard(processor)
AjouterVisualisationsAvancees(dashboard) 

print("\n✓ Dashboard prêt pour utilisation!")
#dashboard.resume_complet()


✓ Dashboard initialisé
  Produits: Ester, Retinol
  Période: 2026-01-01 → 2026-06-11
✅ Visualisations avancées ajoutées au dashboard!

   Nouvelles méthodes disponibles:
   • dashboard.plot_histogramme_tous_produits(mois=3)
   • dashboard.plot_waterfall_mois(mois=3)
   • dashboard.plot_histogramme_jours_mois_v1(mois=3, nom_produit='Ester')
   • dashboard.plot_histogramme_jours_mois_v2(mois=3, nom_produit='Ester')


✓ Dashboard prêt pour utilisation!


In [114]:
# Voir l'évolution temporelle d'un produit
dashboard.afficher_bilan_produit('Ester')


In [115]:
dashboard.plot_histogramme_jours_mois_v2(mois=6, annea=2026, nom_produit='Ester', std=2)

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.3908
  Variation de Stock     : 0.0057
  Stock Entrée.          : -0.0004
  Stock Sortie.          : 0.0052
  Production             : 3.3964
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JUIN 2026 (Termin


  📊 STATISTIQUES - PRODUCTION ESTER PAR JOUR - JUNE 2026
Nombre de jours complets:           11
Production moyenne:                 8.64
Production min/max:                 0.19 / 14.13
Écart-type:                         3.27
Coefficient de variation:           37.9%
Production totale mois:             94.99
TRS mois: (jours complets)          8.64
Jours au-dessus de la moyenne:      6 / 11

🎯 RATIO oee (Production / CMJ en %):
  CMJ (Cible Journalière):            9.50
  oee Moyen (par jour):               90.9%
  oee Min/Max (par jour):             2.0% / 148.7%
  oee Cumulé à date:                  90.9%
  Jours > 100%:                       4 / 11

  Détail oee par jour:
    J01:  113.6% ✅
    J02:   85.6% ⚠️ 
    J03:   94.0% ⚠️ 
    J04:  113.8% ✅
    J05:   77.9% ❌
    J06:  148.7% ✅
    J07:   99.1% ⚠️ 
    J08:   74.7% ❌
    J09:  104.3% ✅
    J10:   86.2% ⚠️ 
    J11:    2.0% ❌



In [116]:
debut = '06-06-2026 02:00:00'
fin = '06-07-2026 02:00:00'
NOP = processor.data.loc[[fin],['ester_nop']].values[0] - processor.data.loc[[debut],['ester_nop']].values[0]
print('Ester NOP   : ', NOP, NOP*2.71)
print('Ester peson : ',processor.data.loc[[fin],['ester_cons']].values[0] - processor.data.loc[[debut],['ester_cons']].values[0] )
print('Ester titre : ', processor.data.loc[[fin],['consommation_Ester']].values[0] - processor.data.loc[[debut],['consommation_Ester']].values[0])
print('Ester Stock : ',processor.data.loc[[fin],['stock_Ester']].values[0], processor.data.loc[[debut],['stock_Ester']].values[0] )
print('Ester Delta : ',processor.data.loc[[fin],['stock_Ester']].values[0] - processor.data.loc[[debut],['stock_Ester']].values[0])
print('Ester conso : ',processor.data.loc[[fin],['conso_delta_Ester']].values[0]- processor.data.loc[[debut],['conso_delta_Ester']].values[0] )
print('Ester prod  : ',processor.data.loc[[fin],['production_Ester']].values[0]- processor.data.loc[[debut],['production_Ester']].values[0] )
processor.data[debut:fin]

Ester NOP   :  [3.] [8.13]
Ester peson :  [4290.]
Ester titre :  [9.52241862]
Ester Stock :  [11.36040689] [6.75235152]
Ester Delta :  [4.60805538]
Ester conso :  [9.52241862]
Ester prod  :  [14.130474]


,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester,consommation_Retinol,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-06-06 02:00:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,430.0,410.0,4042.420,20.6,64.3027,0.32374,3.30051,465.954,787465.0,37.61880,1139.793007,6.752352,998613.508267,0.009393,1139.793007,1.166064,1140.959071,998613.508267,-0.002162,998613.506105
2026-06-06 02:10:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,430.0,410.0,4042.420,20.6,73.3706,6.23565,3.15253,465.098,787465.0,27.93260,1139.793007,7.190287,998613.508267,0.006975,1139.793007,1.604000,1141.397007,998613.508267,-0.004580,998613.503687
2026-06-06 02:20:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,518.0,410.0,4042.420,20.6,78.3551,6.30353,2.69654,491.876,787465.0,21.41770,1139.793007,7.484445,998613.508267,0.005348,1139.793007,1.898157,1141.691164,998613.508267,-0.006207,998613.502060
2026-06-06 02:30:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,660.0,410.0,4042.420,20.6,80.8983,6.26082,2.97003,519.889,787465.0,12.82440,1139.793007,7.676706,998613.508267,0.003202,1139.793007,2.090419,1141.883425,998613.508267,-0.008353,998613.499914
2026-06-06 02:40:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,540.0,1055.0,410.0,832.988,20.6,82.0182,6.27162,3.05909,548.217,788291.0,18.73360,1139.793007,8.013598,998613.678423,0.004678,1139.793007,2.427311,1142.220317,998613.678423,-0.006877,998613.671546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-07 01:20:00+00:00,2734530.0,2195.0,0.0,2219678.0,2188000.0,1310.0,1135.0,410.0,4072.030,20.6,63.8734,6.65234,3.47110,971.130,804186.0,3.91871,1148.116799,10.879373,998616.952793,0.000978,1148.116799,5.293086,1153.409885,998616.952793,-0.010577,998616.942217
2026-06-07 01:30:00+00:00,2734530.0,2195.0,0.0,2219678.0,2188000.0,1310.0,1530.0,410.0,4072.030,20.6,61.8575,6.65234,3.49692,999.811,804186.0,7.97612,1148.116799,10.868757,998616.952793,0.001992,1148.116799,5.282470,1153.399269,998616.952793,-0.009564,998616.943230
2026-06-07 01:40:00+00:00,2734530.0,2195.0,0.0,2219678.0,2188000.0,1310.0,1610.0,410.0,4072.030,20.6,60.4561,1.40625,3.48287,1029.130,804186.0,32.66700,1148.116799,10.851664,998616.952793,0.008157,1148.116799,5.265376,1153.382175,998616.952793,-0.003398,998616.949395


In [117]:
processor.calcul_cumul_journalier(1,6,2026,"Ester")

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
____________________________________________________________


{'Ester': {'consommation': 8.31730239999979,
  'delta_stock': 2.472643477741661,
  'production': 10.789945877741502}}

In [118]:
dashboard.afficher_bilan_journalier(jour=6, mois=6, annee=2026)

____________________________________________________________
BILAN JOURNALIER - 06 JUIN 2026 (Terminé)
Période : du 06/06/2026 à 02:00 au 07/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 9.5224
  Variation de Stock     : 4.6081
  Stock Entrée.          : 1.1661
  Stock Sortie.          : 5.7741
  Production             : 14.1305
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.4445
  Variation de Stock     : 0.0018
  Stock Entrée.          : -0.0022
  Stock Sortie.          : -0.0003
  Production             : 3.4464
____________________________________________________________
____________________________________________________________


In [119]:
dashboard.plot_histogramme_annuee(annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -4.1001
  Stock Entrée.          : 3.0225
  Stock Sortie.          : -1.0776
  Production             : 4.1672
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Mois         Jours    Prod Total      Prod Moy        oee Moy      oee Cumul   
----------------------------------------------------------------------------------------------------
January      29       187.80          6.48            68.2        % 68.2        %
February     26       186.70          7.18            75.6        % 75.6        %
March        28       217.51          7.77            81.8        % 81.8        %
April        30       246.96          8.23            86.7        % 86.7        %
May          31       261.32          8.43            88.7        % 88.7        %
June         11       94.99           8.64            90.9        % 90.9        %



In [120]:
dashboard.plot_histogramme_annee_complet(annea=2026, nom_produit='Ester', std=2)

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -4.1001
  Stock Entrée.          : 3.0225
  Stock Sortie.          : -1.0776
  Production             : 4.1672
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Nombre de jours avec données:       155/365
Production totale année:            1195.29
Production moyenne (jours actifs):  7.71
Production min/max:                 0.03 / 14.13
Écart-type:                         2.83

🎯 RATIO oee:
  CMJ (Cible Journalière):            9.50
  oee Moyen (par jour):               81.2%
  oee Cumulé à date (fin d'année):    34.5%
  Jours > 100% (surproduction):       34 / 155



In [121]:
processor.plot_simple_tag('Hexane')